In [9]:
import pandas as pd
import numpy as np

# For content-based filtering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# For collaborative filtering with Surprise (Matrix Factorization)
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate


In [24]:
movies = pd.read_csv('ml-1m/movies.dat', 
                     sep='::', 
                     header=None, 
                     names=['movieId', 'title', 'genre'], 
                     engine='python', 
                     encoding='latin-1')

# ratings.dat contains: UserID, MovieID, Rating, Timestamp
ratings = pd.read_csv('ml-1m/ratings.dat', 
                      sep='::', 
                      header=None, 
                      names=['userId', 'movieId', 'rating', 'timestamp'],
                      engine='python', 
                      encoding='latin-1')

# users.dat contains: UserID, Gender, Age, Occupation, Zip-code
users = pd.read_csv('ml-1m/users.dat', 
                    sep='::', 
                    header=None, 
                    names=['userId', 'gender', 'age', 'occupation', 'zip_code'], 
                    engine='python', 
                    encoding='latin-1')

In [26]:
movies['metadata'] = movies['genre'].str.replace('|', ' ')

In [28]:
# Build a TF-IDF representation of the genre-based metadata.
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['metadata'])

In [36]:
# Reduce dimensionality with TruncatedSVD (latent content matrix).
n_components = 20  # Adjust the number of components as needed.
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_matrix = svd.fit_transform(tfidf_matrix)


In [38]:
# Compute the cosine similarity matrix among movies based on content features.
cosine_sim = cosine_similarity(svd_matrix, svd_matrix)

In [40]:
# Function to get content-based recommendations given a movie title.
def content_recommendations(title, cosine_sim=cosine_sim, movies=movies, top_n=10):
    try:
        idx = movies[movies['title'] == title].index[0]
    except IndexError:
        print(f"Movie titled '{title}' not found!")
        return None
    # Compute similarity scores and sort them
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]


In [42]:
# 4a. Collaborative Filtering with Surprise (Matrix Factorization)
# Prepare data for Surprise using the ratings dataframe.
reader = Reader(rating_scale=(ratings['rating'].min(), ratings['rating'].max()))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Evaluate the SVD algorithm using cross-validation.
algo = SVD()
cross_validate(algo, data, measures=['RMSE'], cv=5, verbose=True)

Evaluating RMSE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8736  0.8783  0.8728  0.8745  0.8721  0.8742  0.0022  
Fit time          16.06   16.89   16.95   15.74   16.17   16.36   0.48    
Test time         2.49    2.83    2.53    2.38    2.63    2.57    0.15    


{'test_rmse': array([0.87358102, 0.87825705, 0.87279965, 0.8745386 , 0.87206514]),
 'fit_time': (16.05778694152832,
  16.888734817504883,
  16.947438955307007,
  15.737410306930542,
  16.16849660873413),
 'test_time': (2.4922070503234863,
  2.8303143978118896,
  2.5263478755950928,
  2.382795810699463,
  2.631420373916626)}

In [44]:
# Train on the full dataset.
trainset = data.build_full_trainset()
algo.fit(trainset)

In [46]:
# Function to recommend movies for a given user based on predicted ratings.
def collaborative_recommendations(userId, movies, ratings, algo, top_n=10):
    movie_ids = movies['movieId'].unique()
    user_rated = ratings[ratings['userId'] == userId]['movieId'].tolist()
    recommendations = []
    for movie_id in movie_ids:
        if movie_id not in user_rated:
            pred = algo.predict(userId, movie_id)
            recommendations.append((movie_id, pred.est))
    recommendations.sort(key=lambda x: x[1], reverse=True)
    top_movie_ids = [x[0] for x in recommendations[:top_n]]
    return movies[movies['movieId'].isin(top_movie_ids)][['movieId', 'title']]


In [50]:
popularity = ratings.groupby('movieId')['rating'].mean().reset_index()
popularity = pd.merge(popularity, movies[['movieId', 'title']], on='movieId', how='left')
popularity = popularity.sort_values('rating', ascending=False)
print("Top Popular Movies:")
print(popularity.head(10))

Top Popular Movies:
      movieId  rating                                      title
926       989     5.0  Schlafes Bruder (Brother of Sleep) (1995)
3635     3881     5.0                   Bittersweet Motel (2000)
1652     1830     5.0                    Follow the Bitch (1998)
3152     3382     5.0                     Song of Freedom (1936)
744       787     5.0         Gate of Heavenly Peace, The (1995)
3054     3280     5.0                           Baby, The (1973)
3367     3607     5.0                   One Little Indian (1973)
3010     3233     5.0                       Smashing Time (1967)
2955     3172     5.0                    Ulysses (Ulisse) (1954)
3414     3656     5.0                               Lured (1947)


In [52]:
def hybrid_recommendations(userId, liked_title, movies, ratings, algo, cosine_sim, 
                           weight_content=0.5, weight_collab=0.5, top_n=10):
    # Get collaborative filtering scores for movies not yet rated by the user.
    movie_ids = movies['movieId'].unique()
    user_rated = ratings[ratings['userId'] == userId]['movieId'].tolist()
    collab_scores = {}
    for movie_id in movie_ids:
        if movie_id not in user_rated:
            pred = algo.predict(userId, movie_id)
            collab_scores[movie_id] = pred.est

    # Get the content-based score relative to the liked movie.
    try:
        liked_idx = movies[movies['title'] == liked_title].index[0]
    except IndexError:
        print(f"Movie titled '{liked_title}' not found!")
        return None
    content_scores = dict(enumerate(cosine_sim[liked_idx]))

    # Combine the two scores. (Normalization might be needed in production.)
    hybrid_scores = {}
    for idx, movie in movies.iterrows():
        if movie['movieId'] in collab_scores:
            hybrid_scores[movie['movieId']] = weight_content * content_scores.get(idx, 0) + \
                                                weight_collab * collab_scores[movie['movieId']]
    hybrid_sorted = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    top_movie_ids = [x[0] for x in hybrid_sorted[:top_n]]
    return movies[movies['movieId'].isin(top_movie_ids)][['movieId', 'title']]


In [54]:
# Example hybrid recommendation for user 1 based on liking 'Toy Story (1995)'
print("Hybrid Recommendations for userId=1 with liked movie 'Toy Story (1995)':")
hybrid_recs = hybrid_recommendations(userId=1, 
                                     liked_title='Toy Story (1995)', 
                                     movies=movies, 
                                     ratings=ratings, 
                                     algo=algo, 
                                     cosine_sim=cosine_sim, 
                                     top_n=10)
print(hybrid_recs)

Hybrid Recommendations for userId=1 with liked movie 'Toy Story (1995)':
      movieId                                        title
360       364                        Lion King, The (1994)
592       596                             Pinocchio (1940)
1010     1023  Winnie the Pooh and the Blustery Day (1968)
1132     1148                   Wrong Trousers, The (1993)
1205     1223                      Grand Day Out, A (1992)
1262     1282                              Fantasia (1940)
2009     2078                      Jungle Book, The (1967)
2016     2085                        101 Dalmatians (1961)
2068     2137                       Charlotte's Web (1973)
2070     2139                   Secret of NIMH, The (1982)


## Sample

In [60]:
#Popularity-Based Recommendation
print("Top 10 Popular Movies:")
print(popularity[['title', 'rating']].head(10))


Top 10 Popular Movies:
                                          title  rating
926   Schlafes Bruder (Brother of Sleep) (1995)     5.0
3635                   Bittersweet Motel (2000)     5.0
1652                    Follow the Bitch (1998)     5.0
3152                     Song of Freedom (1936)     5.0
744          Gate of Heavenly Peace, The (1995)     5.0
3054                           Baby, The (1973)     5.0
3367                   One Little Indian (1973)     5.0
3010                       Smashing Time (1967)     5.0
2955                    Ulysses (Ulisse) (1954)     5.0
3414                               Lured (1947)     5.0


In [62]:
#Content-Based Filtering
title = 'Toy Story (1995)'
print(f"Content-Based Recommendations for '{title}':")
print(content_recommendations(title, top_n=10))


Content-Based Recommendations for 'Toy Story (1995)':
1050            Aladdin and the King of Thieves (1996)
2072                          American Tail, An (1986)
2073        American Tail: Fievel Goes West, An (1991)
2285                         Rugrats Movie, The (1998)
2286                              Bug's Life, A (1998)
3045                                Toy Story 2 (1999)
3542                             Saludos Amigos (1943)
3682                                Chicken Run (2000)
3685    Adventures of Rocky and Bullwinkle, The (2000)
12                                        Balto (1995)
Name: title, dtype: object


In [66]:
# Collaborative Filtering
user_id = 1
print(f"Collaborative Filtering Recommendations for user {user_id}:")
print(collaborative_recommendations(user_id, movies, ratings, algo, top_n=10))


Collaborative Filtering Recommendations for user 1:
      movieId                             title
108       110                 Braveheart (1995)
315       318  Shawshank Redemption, The (1994)
892       904                Rear Window (1954)
908       920         Gone with the Wind (1939)
912       924      2001: A Space Odyssey (1968)
941       953      It's a Wonderful Life (1946)
1186     1204         Lawrence of Arabia (1962)
1252     1272                     Patton (1970)
1262     1282                   Fantasia (1940)
2836     2905                    Sanjuro (1962)


In [68]:
#Matrix Factorization
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8736  0.8744  0.8722  0.8730  0.8753  0.8737  0.0011  
MAE (testset)     0.6846  0.6865  0.6851  0.6865  0.6871  0.6859  0.0010  
Fit time          15.16   14.55   15.50   15.99   14.91   15.22   0.49    
Test time         2.59    2.23    2.56    2.47    2.33    2.44    0.14    


{'test_rmse': array([0.87357904, 0.87443056, 0.87216162, 0.87301898, 0.87532962]),
 'test_mae': array([0.68457329, 0.68650689, 0.68507411, 0.68652922, 0.68705277]),
 'fit_time': (15.157147407531738,
  14.554793357849121,
  15.4954833984375,
  15.994519710540771,
  14.909129619598389),
 'test_time': (2.5905048847198486,
  2.2262566089630127,
  2.5600101947784424,
  2.4661922454833984,
  2.3331458568573)}

In [69]:
#Hybrid Recommendation
user_id = 1
liked_movie = 'Toy Story (1995)'
print(f"Hybrid Recommendations for user {user_id} with liked movie '{liked_movie}':")
print(hybrid_recommendations(user_id, liked_movie, movies, ratings, algo, cosine_sim, top_n=10))


Hybrid Recommendations for user 1 with liked movie 'Toy Story (1995)':
      movieId                                        title
33         34                                  Babe (1995)
592       596                             Pinocchio (1940)
1010     1023  Winnie the Pooh and the Blustery Day (1968)
1132     1148                   Wrong Trousers, The (1993)
1205     1223                      Grand Day Out, A (1992)
2009     2078                      Jungle Book, The (1967)
2012     2081                   Little Mermaid, The (1989)
2068     2137                       Charlotte's Web (1973)
3352     3421                          Animal House (1978)
3360     3429                     Creature Comforts (1990)
